# 📱 Mobile Price Classification (Telefon Narx Kategoriyasini Aniqlash)

- **Kutubxonalar:** `Scikit-learn`, `Pandas`, `kagglehub`
- **Dataset manbasi:** [Kaggle - Mobile Price Classification](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification)
- **Tillar:** 🇺🇿 O'zbekcha, 🇷🇺 Русский, 🇬🇧 English

Ushbu loyihada mobil telefonlarning texnik parametrlari (RAM, batareya hajmi, kameralar, ichki xotira, protsessor) asosida ularning narx toifasini (0: Arzon, 1: O'rtacha, 2: Qimmat, 3: Flagman) bashorat qilish tizimi ishlab chiqiladi.

In [ ]:
# 1. Kerakli kutubxonalarni import qilish
import os
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 📥 2. Kaggle'dan Datasetni Yuklash va Pandas orqali O'qish

In [ ]:
path = kagglehub.dataset_download("iabhishekofficial/mobile-price-classification")
print("Path to dataset files:", path)

df_train = pd.read_csv(os.path.join(path, "train.csv"))
df_test = pd.read_csv(os.path.join(path, "test.csv"))

print(f"Train shakli: {df_train.shape}")
print(f"Test shakli: {df_test.shape}")
df_train.head()

## 📊 3. Pandas Bilan Statistik Tahlil (EDA) va Korrelyatsiya

In [ ]:
print("Missing values:", df_train.isnull().sum().sum())
print("\nNarx toifalari taqsimoti:")
print(df_train['price_range'].value_counts())

# Korrelyatsiyalar (RAM narxga eng kuchli ta'sir ko'rsatadi)
corrs = df_train.corr()['price_range'].sort_values(ascending=False)
print("\nNarx toifasiga ta'sir qiluvchi asosiy omillar:")
print(corrs.head(6))

## 🤖 4. Scikit-learn: Modellarni O'qitish va Baholash

In [ ]:
X = df_train.drop('price_range', axis=1)
y = df_train['price_range']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Support Vector Classifier": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='linear', random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    print(f"{name}: Aniqlik = {acc*100:.2f}%")

## 🎯 5. Yangi Telefon Parametrlari Asosida Narx Toifasini Bashorat Qilish

In [ ]:
best_model = models["Logistic Regression"]

def predict_price_category(ram_mb, battery_power, primary_camera=16, front_camera=8, storage=64):
    row = X.mean().to_dict()
    row['ram'] = ram_mb
    row['battery_power'] = battery_power
    row['pc'] = primary_camera
    row['fc'] = front_camera
    row['int_memory'] = storage
    
    df_row = pd.DataFrame([row])[X.columns]
    cat = int(best_model.predict(df_row)[0])
    
    labels = {
        0: "0 - Hamyonbop / Arzon narx (Low Cost)",
        1: "1 - O'rta-quyi toifa (Medium Cost)",
        2: "2 - O'rta-yuqori toifa (High Cost)",
        3: "3 - Flagman / Juda qimmat (Very High Cost)"
    }
    return labels[cat]

# Sinov namunalari
print("1. Flagman telefon:", predict_price_category(ram_mb=3900, battery_power=4000, primary_camera=24))
print("2. O'rta toifa telefon:", predict_price_category(ram_mb=2300, battery_power=2500, primary_camera=12))
print("3. Arzon telefon:", predict_price_category(ram_mb=800, battery_power=1000, primary_camera=5))